# Reconhecimento de Gestos Customizados com OpenCV e MediaPipe

Este notebook integra o modelo de Machine Learning treinado anteriormente (`gesture_model.pkl`) com a captura em tempo real do MediaPipe. O MediaPipe extrai as coordenadas das mãos e o modelo Random Forest classifica o gesto.

In [9]:
import cv2
import mediapipe as mp
import numpy as np
import os
import joblib
import pandas as pd
import sklearn

# Inicializa utilitários de desenho
mp_hands_connections = mp.tasks.vision.HandLandmarksConnections
mp_drawing = mp.tasks.vision.drawing_utils
mp_drawing_styles = mp.tasks.vision.drawing_styles

# Caminhos para os modelos
MP_TASK_PATH = "gesture_recognizer.task" 
CUSTOM_MODEL_PATH = "gesture_model.pkl"
LABEL_ENCODER_PATH = "label_encoder.pkl"

def main():
    # Verifica se os modelos existem
    if not all(os.path.exists(p) for p in [MP_TASK_PATH, CUSTOM_MODEL_PATH, LABEL_ENCODER_PATH]):
        print("Erro: Um ou mais arquivos de modelo não foram encontrados (.task, .pkl).")
        return

    # Carrega o modelo customizado e o encoder
    # O model foi treinado com nomes de colunas (x0, y0, z0...). 
    # Alguns modelos do sklearn emitem avisos se passarmos apenas o array.
    clf = joblib.load(CUSTOM_MODEL_PATH)
    label_encoder = joblib.load(LABEL_ENCODER_PATH)
    
    # Nomes das colunas usados no treinamento
    feature_names = []
    for i in range(21):
        feature_names.extend([f'x{i}', f'y{i}', f'z{i}'])
    
    print(f"Modelo customizado carregado. Classes: {label_encoder.classes_}")

    # Inicializa o modelo do MediaPipe Tasks
    BaseOptions = mp.tasks.BaseOptions
    GestureRecognizer = mp.tasks.vision.GestureRecognizer
    GestureRecognizerOptions = mp.tasks.vision.GestureRecognizerOptions
    VisionRunningMode = mp.tasks.vision.RunningMode

    # Configurações do MediaPipe
    options = GestureRecognizerOptions(
        base_options=BaseOptions(model_asset_path=MP_TASK_PATH),
        running_mode=VisionRunningMode.VIDEO,
        num_hands=2,
        min_hand_detection_confidence=0.5,
        min_hand_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )

    cap = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

    print("\nIniciando reconhecimento... Pressione 'q' para sair.")

    with GestureRecognizer.create_from_options(options) as recognizer:
        while cap.isOpened():
            success, frame = cap.read()
            if not success:
                break

            frame = cv2.flip(frame, 1)
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            timestamp_ms = int(cv2.getTickCount() / cv2.getTickFrequency() * 1000)
            
            # Extrai landmarks usando MediaPipe
            recognition_result = recognizer.recognize_for_video(mp_image, timestamp_ms)

            if recognition_result.hand_landmarks:
                for i, hand_landmarks in enumerate(recognition_result.hand_landmarks):
                    # 1. Extrai coordenadas para o modelo customizado (mesmo formato do treino)
                    landmarks_row = []
                    for lm in hand_landmarks:
                        landmarks_row.extend([lm.x, lm.y, lm.z])
                    
                    # Transforma em DataFrame para manter nomes das colunas e evitar avisos do sklearn
                    input_df = pd.DataFrame([landmarks_row], columns=feature_names)
                    
                    # 2. Predição
                    prediction = clf.predict(input_df)
                    gesture_name = label_encoder.inverse_transform(prediction)[0]
                    
                    # Probabilidade
                    proba = np.max(clf.predict_proba(input_df))

                    # 3. Visualização
                    # Mão (Esquerda/Direita)
                    hand_label = recognition_result.handedness[i][0].category_name
                    
                    # Desenha as landmarks no frame
                    mp_drawing.draw_landmarks(
                        frame,
                        hand_landmarks,
                        mp_hands_connections.HAND_CONNECTIONS,
                        mp_drawing_styles.get_default_hand_landmarks_style(),
                        mp_drawing_styles.get_default_hand_connections_style()
                    )

                    display_text = f"{hand_label}: {gesture_name} ({proba:.2f})"
                    cv2.putText(frame, display_text, (20, 50 + (i * 40)),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

            cv2.imshow('Custom Gesture Recognition', frame)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

Modelo customizado carregado. Classes: ['coracao' 'hangloose' 'joinha' 'nada' 'ok' 'pare' 'paz' 'rock'
 'va se foder']

Iniciando reconhecimento... Pressione 'q' para sair.
